# Personne 2 — Décision métier, coût des erreurs et validation

**Mission individuelle :** définir comment mesurer une décision de crédit. Étudier les erreurs, établir une règle métier de référence et proposer un protocole de validation. Vous ne devez pas entraîner le modèle final.

**Dataset Kaggle requis :** ajoutez le jeu de données du sujet avec `train.csv` et `a_decider.csv` (et si disponible `HomeCredit_columns_description.csv`).

**Règle commune :** ne jamais utiliser `SK_ID_CURR` comme variable prédictive et ne jamais inventer la cible des lignes à prédire.

## 1. Charger les données

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, os, time, warnings
warnings.filterwarnings("ignore")
def trouver(nom):
    candidats = [Path("/kaggle/input"), Path.cwd(), Path.cwd().parent / "Sujet_03_Home_Credit", Path("/content")]
    for racine in candidats:
        if racine.exists():
            trouve = list(racine.rglob(nom))
            if trouve: return trouve[0]
    raise FileNotFoundError(f"{nom} introuvable. Ajoutez le Dataset Kaggle avec Add Input.")
TRAIN_PATH = trouver("train.csv")
TEST_PATH = trouver("a_decider.csv")
print("Train :", TRAIN_PATH)
print("À prédire :", TEST_PATH)

train = pd.read_csv(TRAIN_PATH)
y = train["TARGET"].astype(int)
print(train.shape, y.mean())

## 2. Comprendre les deux erreurs

- **Faux négatif (FN)** : le modèle prédit « pas de défaut » alors que `TARGET=1` ; un crédit risqué est accordé.
- **Faux positif (FP)** : le modèle prédit « défaut » alors que `TARGET=0` ; un bon client est refusé.

Le sujet demande de choisir un rapport de coûts. Les coûts ci-dessous sont des **hypothèses de travail**, à discuter et justifier avec l'enseignant, pas des valeurs données par les fichiers.

In [ ]:
COUT_FN = 5.0  # coût hypothétique d'un défaut accordé
COUT_FP = 1.0  # coût hypothétique d'un bon client refusé
print("Coût FN :", COUT_FN, "| Coût FP :", COUT_FP)

## 3. Construire une règle métier très simple (plancher)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
X = train.drop(columns=["TARGET", "SK_ID_CURR"])
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
# Règle naïve : prédire la classe majoritaire (aucun défaut)
pred_plancher = np.zeros(len(y_valid), dtype=int)
cm = confusion_matrix(y_valid, pred_plancher, labels=[0,1])
tn, fp, fn, tp = cm.ravel()
cout = (COUT_FN*fn + COUT_FP*fp)/len(y_valid)
print("Matrice [TN, FP, FN, TP] :")
print(cm)
print("Coût moyen hypothétique par dossier :", round(cout, 5))
print(classification_report(y_valid, pred_plancher, zero_division=0))

## 4. Tester l'effet de plusieurs seuils sans modèle

Cette cellule illustre la logique du seuil avec un score fictif constant : elle ne constitue pas une performance de modèle. La personne 3 fournira ensuite les probabilités prédites.

In [ ]:
def evaluer_seuil(y_vrai, probabilites, seuil, cout_fn=COUT_FN, cout_fp=COUT_FP):
    pred = (np.asarray(probabilites) >= seuil).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_vrai, pred, labels=[0,1]).ravel()
    return {"seuil": seuil, "TN":tn, "FP":fp, "FN":fn, "TP":tp,
            "cout_total": cout_fn*fn + cout_fp*fp,
            "cout_moyen": (cout_fn*fn + cout_fp*fp)/len(y_vrai)}
print("Fonction d'évaluation prête.")

## 5. Protocole de validation recommandé

Proposition commune à utiliser dans les autres notebooks :
- séparation stratifiée 80 % entraînement / 20 % validation ;
- `random_state=42` pour que chaque personne compare les mêmes lignes ;
- aucune information de `a_decider.csv` dans l'entraînement ou le choix du seuil ;
- toutes les étapes d'imputation et d'encodage sont ajustées uniquement sur le sous-ensemble d'entraînement ;
- présenter matrice de confusion, rappel de la classe défaut, précision et coût métier.

Le coût final dépend des hypothèses `COUT_FN` et `COUT_FP`. Il faut tester plusieurs rapports de coûts et expliquer le choix.

## 6. Livrables

In [ ]:
rapport = pd.DataFrame([{
    "COUT_FN": COUT_FN, "COUT_FP": COUT_FP,
    "random_state": 42, "test_size": 0.2,
    "taux_defaut_train": y.mean(),
    "cout_moyen_plancher_validation": cout
}])
rapport.to_csv("/kaggle/working/protocole_decision.csv", index=False)
print("Fichier créé : /kaggle/working/protocole_decision.csv")

## Travail à rendre
1. Justifier le rapport de coûts entre défaut accordé et bon client refusé.
2. Expliquer pourquoi l'accuracy seule est trompeuse avec environ 8 % de défauts.
3. Présenter le plancher et le découpage de validation.
4. Transmettre les valeurs de coûts retenues à l'équipe.